# WR503 上手／下手震動比較：2026-03-26 vs 2026-08-27

比較相同條件的原始／濾波前量測，上手只和上手比、下手只和下手比。所有時間序列圖使用完全相同的 y 軸範圍與刻度，紅色虛線為 ±0.3 g 門檻。

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
DATASETS = {
    "2026-03-26 upper/raw": Path(r"C:\ITRI\歐陽\SAA\WR503\震動量測\20260326\上手\RecData--20260326133043上手.csv"),
    "2026-08-27 upper J5/raw": Path(r"C:\ITRI\歐陽\SAA\WR503\震動量測\20260827_vib test\濾波前\上手J5\RecData--20260827115016.csv"),
    "2026-03-26 lower/raw": Path(r"C:\ITRI\歐陽\SAA\WR503\震動量測\20260326\下手\RecData--20260326133043下手.csv"),
    "2026-08-27 lower J4/raw": Path(r"C:\ITRI\歐陽\SAA\WR503\震動量測\20260827_vib test\濾波前\下手J4\RecData--20260827115019.csv"),
}
GROUPS = {
    "Upper gripper": [label for label in DATASETS if "upper" in label],
    "Lower gripper": [label for label in DATASETS if "lower" in label],
}
THRESHOLD_G = 0.3
Y_PADDING_RATIO = 0.03  # Add 3% beyond the largest absolute value
Y_TICK_STEP_G = 0.2     # Common, readable tick spacing
DISPLAY_MAX_POINTS = 60_000

missing = [str(path) for path in DATASETS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing CSV:\n" + "\n".join(missing))


In [ ]:
def load_recdata(path):
    with path.open("r", encoding="utf-8-sig", errors="replace") as stream:
        preview = [stream.readline() for _ in range(30)]
    header_row = next(
        i for i, line in enumerate(preview)
        if "time" in line.lower() and "," in line
    )
    metadata = {}
    for line in preview[:header_row]:
        if ":" in line:
            key, value = line.split(":", 1)
            metadata[key.strip()] = value.strip()
    frame = pd.read_csv(path, skiprows=header_row)
    frame.columns = frame.columns.str.strip()
    frame = frame.rename(columns={"Time": "time_s", "X-axis": "X", "Y-axis": "Y", "Z-axis": "Z"})
    required = ["time_s", "X", "Y", "Z"]
    if not set(required).issubset(frame.columns):
        raise ValueError(f"Unexpected columns in {path}: {list(frame.columns)}")
    frame = frame[required].apply(pd.to_numeric, errors="coerce").dropna()
    frame = frame.sort_values("time_s").drop_duplicates("time_s")
    return frame, metadata

series = {}
metadata = {}
for label, path in DATASETS.items():
    series[label], metadata[label] = load_recdata(path)
    print(f"{label}: {len(series[label]):,} samples, {series[label].time_s.iloc[-1]:.3f} s, metadata={metadata[label]}")

# Derive one compact symmetric y scale from the largest absolute value in all six datasets.
global_abs_max = max(frame[["X", "Y", "Z"]].abs().to_numpy().max() for frame in series.values())
Y_LIMIT_G = np.ceil(global_abs_max * (1 + Y_PADDING_RATIO) / 0.1) * 0.1
Y_TICKS_G = np.arange(-Y_LIMIT_G, Y_LIMIT_G + Y_TICK_STEP_G / 2, Y_TICK_STEP_G)
print(f"Common y range: {-Y_LIMIT_G:.1f} to {Y_LIMIT_G:.1f} g; ticks every {Y_TICK_STEP_G:.1f} g (global max |g|={global_abs_max:.6f})")


In [ ]:
rows = []
for label, frame in series.items():
    dt = frame.time_s.diff().dropna().median()
    for axis in ["X", "Y", "Z"]:
        values = frame[axis].to_numpy()
        absolute = np.abs(values)
        exceeded = absolute >= THRESHOLD_G  # Vendor requirement is strictly < 0.3 g
        transitions = np.diff(np.r_[False, exceeded, False].astype(np.int8))
        run_lengths = np.flatnonzero(transitions == -1) - np.flatnonzero(transitions == 1)
        event_count = len(run_lengths)
        rows.append({
            "dataset": label,
            "axis": axis,
            "samples": len(values),
            "sampling_rate_Hz": 1 / dt,
            "mean_g": values.mean(),
            "std_g": values.std(ddof=1),
            "RMS_g": np.sqrt(np.mean(values ** 2)),
            "p95_abs_g": np.percentile(absolute, 95),
            "p99_abs_g": np.percentile(absolute, 99),
            "p99.9_abs_g": np.percentile(absolute, 99.9),
            "max_abs_g": absolute.max(),
            "min_g": values.min(),
            "max_g": values.max(),
            "within_0.3g_pct": 100 * np.mean(absolute < THRESHOLD_G),
            "outside_0.3g_count": np.count_nonzero(exceeded),
            "outside_0.3g_pct": 100 * np.mean(exceeded),
            "outside_0.3g_seconds": np.count_nonzero(exceeded) * dt,
            "exceedance_events": event_count,
            "exceedance_events_per_min": event_count / (len(values) * dt / 60),
            "max_exceedance_duration_ms": (run_lengths.max() * dt * 1000) if event_count else 0.0,
        })
metrics = pd.DataFrame(rows).set_index(["dataset", "axis"])
display(metrics.round(6))


In [ ]:
# Rows are axes and columns are dates/runs; upper and lower grippers are separate figures.
axes_names = ["X", "Y", "Z"]
global_duration = max(frame.time_s.iloc[-1] for frame in series.values())
for gripper, labels in GROUPS.items():
    fig, axs = plt.subplots(3, len(labels), figsize=(18, 10), sharex=True, sharey=True, squeeze=False)
    for col, label in enumerate(labels):
        frame = series[label]
        step = max(1, int(np.ceil(len(frame) / DISPLAY_MAX_POINTS)))
        view = frame.iloc[::step]
        for row, axis in enumerate(axes_names):
            ax = axs[row, col]
            ax.plot(view.time_s, view[axis], linewidth=0.45, color=f"C{col}")
            ax.axhline(THRESHOLD_G, color="red", linestyle="--", linewidth=1.1)
            ax.axhline(-THRESHOLD_G, color="red", linestyle="--", linewidth=1.1)
            ax.set_ylim(-Y_LIMIT_G, Y_LIMIT_G)
            ax.set_yticks(Y_TICKS_G)
            ax.set_xlim(0, global_duration)
            if row == 0:
                ax.set_title(label)
            if col == 0:
                ax.set_ylabel(f"{axis} acceleration (g)")
            if row == 2:
                ax.set_xlabel("Time (s)")
    fig.suptitle(f"{gripper} raw vibration comparison — common y scale, threshold ±0.3 g", y=1.01)
    fig.tight_layout()
    plt.show()


In [ ]:
# Overlay view makes amplitude and timing differences easier to compare directly.
for gripper, labels in GROUPS.items():
    fig, axs = plt.subplots(3, 1, figsize=(16, 10), sharex=True, sharey=True)
    for axis, ax in zip(axes_names, axs):
        for label in labels:
            frame = series[label]
            step = max(1, int(np.ceil(len(frame) / DISPLAY_MAX_POINTS)))
            view = frame.iloc[::step]
            ax.plot(view.time_s, view[axis], linewidth=0.45, alpha=0.7, label=label)
        ax.axhline(THRESHOLD_G, color="red", linestyle="--", linewidth=1.1, label="±0.3 g threshold" if axis == "X" else None)
        ax.axhline(-THRESHOLD_G, color="red", linestyle="--", linewidth=1.1)
        ax.set_ylim(-Y_LIMIT_G, Y_LIMIT_G)
        ax.set_yticks(Y_TICKS_G)
        ax.set_ylabel(f"{axis} acceleration (g)")
    axs[0].legend(loc="upper right", fontsize=8)
    axs[-1].set_xlabel("Time (s)")
    fig.suptitle(f"{gripper} raw vibration overlay — common y scale")
    fig.tight_layout()
    plt.show()


In [ ]:
threshold_summary = metrics[["RMS_g", "min_g", "max_g", "outside_0.3g_count", "outside_0.3g_pct"]].copy()
display(threshold_summary.round(6))
ax = threshold_summary["outside_0.3g_pct"].unstack("axis").plot.bar(figsize=(14, 5), width=0.8)
ax.set_ylabel("Samples outside ±0.3 g (%)")
ax.set_xlabel("")
ax.set_title("Threshold exceedance rate by dataset and axis")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


## Vendor-oriented quantitative improvement

The vendor criterion is strictly **|acceleration| < 0.3 g while transporting 265 wafers**. These recordings were made with 250 wafers, so the tables below quantify improvement after the July mechanical changes but **do not constitute 265-wafer acceptance evidence**. Positive improvement means the August value is lower/better; a negative value means regression.

In [ ]:
LOWER_IS_BETTER = [
    "RMS_g", "p99_abs_g", "p99.9_abs_g", "max_abs_g",
    "outside_0.3g_pct", "exceedance_events_per_min", "max_exceedance_duration_ms",
]
comparison_rows = []
for gripper, labels in GROUPS.items():
    current_label = next(label for label in labels if "2026-08-27" in label)
    baseline_labels = [label for label in labels if label != current_label]
    baseline_sets = [(label, metrics.loc[label]) for label in baseline_labels]
    if len(baseline_labels) > 1:
        baseline_sets.append(("Mean of baselines", pd.concat([metrics.loc[label] for label in baseline_labels]).groupby(level=0).mean()))
    current = metrics.loc[current_label]
    for baseline_name, baseline in baseline_sets:
        for axis in ["X", "Y", "Z"]:
            row = {"gripper": gripper, "baseline": baseline_name, "axis": axis}
            for metric in LOWER_IS_BETTER:
                old, new = baseline.loc[axis, metric], current.loc[axis, metric]
                row[f"{metric}__baseline"] = old
                row[f"{metric}__august"] = new
                row[f"{metric}__improvement_pct"] = 100 * (old - new) / old if old != 0 else np.nan
            comparison_rows.append(row)
improvement = pd.DataFrame(comparison_rows).set_index(["gripper", "baseline", "axis"])

for metric in LOWER_IS_BETTER:
    print(f"\n{metric}")
    display(improvement[[f"{metric}__baseline", f"{metric}__august", f"{metric}__improvement_pct"]]
            .rename(columns={f"{metric}__baseline": "benchmark", f"{metric}__august": "2026-08-27", f"{metric}__improvement_pct": "improvement_%"})
            .round(4))

current_labels = [label for label in metrics.index.get_level_values(0).unique() if "2026-08-27" in label]
acceptance = metrics.loc[current_labels, ["within_0.3g_pct", "outside_0.3g_count", "outside_0.3g_seconds", "max_abs_g"]].copy()
acceptance["all_samples_strictly_below_0.3g"] = acceptance["outside_0.3g_count"].eq(0)
print("August 250-wafer threshold evidence (not a 265-wafer acceptance test):")
display(acceptance.round(6))

for metric in ["RMS_g", "p99_abs_g", "outside_0.3g_pct", "exceedance_events_per_min"]:
    chart = improvement[f"{metric}__improvement_pct"].unstack("axis")
    ax = chart.plot.bar(figsize=(14, 5), width=0.8)
    ax.axhline(0, color="black", linewidth=0.9)
    ax.set_ylabel("Improvement vs benchmark (%)")
    ax.set_title(f"{metric}: positive = improvement, negative = regression")
    ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    plt.show()
